# 05 · Encode & scale (Issue #4), then connect to a baseline model

Uses **Sama's `encode_scale`** (`src/pipeline/step5_encode_scale.py`) on the feature table from notebook 04.

```
01_train_base_eda          (Skylar, EDA only)
02_static_0_preprocessing  (Skylar, EDA only)
        │  findings / rules
        ▼
03_clean_missing_outliers   Issue #3   raw CSV ─► data/interim/applicant_{train,test}.parquet
        ▼
04_feature_engineering      Issue #6   ─► data/processed/features_{train,test}.parquet
        ▼
05_encode_scale             Issue #4   ─► data/processed/model_ready_{fit,valid,test}.parquet
        ▼
models (October)
```
Run 03 → 04 → 05 in order; each notebook reads the file the previous one saved.

**Input:** `data/processed/features_{train,test}.parquet`
**Output:** `data/processed/model_ready_{fit,valid,test}.parquet`: all-numeric, imputed, scaled, one-hot encoded.

**Why this runs *after* feature engineering:** the imputer, scaler and one-hot encoder must be fitted **once**, on the final
set of columns, using training weeks only. Aggregations (sums, means per applicant) also need raw, unscaled values.

## Why encoding & scaling comes last (right before models)

```
01 base EDA ─┐
02 static_0 EDA ─┴─► 03 clean ─► 04 feature engineering ─► 05 encode & scale ─► models
```

- **It's the only step that learns from the data** (medians, means/std, category lists). These must be fitted on **training weeks
  < 80 only** and then reused unchanged on validation and test. Otherwise information from later weeks leaks into training.
- **It needs the final set of columns.** Running after 04 means every engineered feature is imputed, flagged if missing, and scaled
  in the same, single fitted `preprocessor`.
- **Models need all-numeric, gap-free input.** Logistic regression can't take text categories or NaNs, and scaling keeps its
  coefficients comparable. (Tree models don't need the scaling, but it doesn't hurt them.)
- **Test is transformed, never fitted.** `encode_scale(test, preprocessor=preprocessor)` only applies what was learned from training,
  so test always ends up with exactly the same 106 columns.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent                      # notebooks/ -> repo root
RAW = ROOT / "data" / "raw" / "csv_files"     # unzipped dataset (git-ignored, see README)
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# Train = weeks 0-91, real test = weeks 92-107. Weeks 80-91 are our out-of-time
# validation set, so anything "learned" from data (caps, medians, scalers)
# is fitted on weeks < 80 only.
VALID_START_WEEK = 80

import sys
sys.path.insert(0, str(ROOT))
from src.pipeline.step5_encode_scale import encode_scale

## 1. Out-of-time split

The technical report (§7) asks for a **time-based** split, not a random one. Notebook 02's `train_test_split` was exploratory.
- **fit** = weeks 0-79: everything is learned here
- **valid** = weeks 80-91: stands in for the later test weeks (92-107)

In [2]:
train = pd.read_parquet(PROCESSED / "features_train.parquet")
test = pd.read_parquet(PROCESSED / "features_test.parquet")

fit_df = train[train["WEEK_NUM"] < VALID_START_WEEK]
valid_df = train[train["WEEK_NUM"] >= VALID_START_WEEK]
print(f"fit {len(fit_df):,} | valid {len(valid_df):,} | test {len(test):,}")

fit 869,559 | valid 130,441 | test 200,000


## 2. Run Sama's encode_scale

Columns that are **not** model inputs:
- `date_decision`, `birth_259D`: raw dates (already turned into `age_years`; a raw date would be one-hot encoded as ~18k categories)
- `WEEK_NUM`: kept only for splitting. Test weeks (92+) lie outside the training range, so it can't generalize.

`encode_scale` itself excludes `case_id` and `target`. It is **fitted on `fit_df` only**, and the returned `preprocessor`
is reused unchanged on valid and test.

In [3]:
NOT_FEATURES = ["date_decision", "birth_259D", "WEEK_NUM"]

X_fit, preprocessor = encode_scale(fit_df.drop(columns=NOT_FEATURES))
X_valid, _ = encode_scale(valid_df.drop(columns=NOT_FEATURES), preprocessor=preprocessor)
X_test, _ = encode_scale(test.drop(columns=NOT_FEATURES), preprocessor=preprocessor)

names = pd.Series(X_fit.columns)
print("X_fit", X_fit.shape, "| X_valid", X_valid.shape, "| X_test", X_test.shape)
print("scaled numeric columns :", (names.str.startswith("numeric__") & ~names.str.contains("missingindicator")).sum())
print("missing-value flags    :", names.str.contains("missingindicator").sum())
print("one-hot columns        :", names.str.startswith("categorical__").sum())
names[names.str.startswith("categorical__")].head(8).tolist()

X_fit (869559, 106) | X_valid (130441, 106) | X_test (200000, 106)
scaled numeric columns : 52
missing-value flags    : 33
one-hot columns        : 21


['categorical__education_927M_10795bac',
 'categorical__education_927M_242b264b',
 'categorical__education_927M_2d4a2bc4',
 'categorical__education_927M_6def22f0',
 'categorical__education_927M_f937ecaa',
 'categorical__maritalstatus_703M_b58630a7',
 'categorical__maritalstatus_703M_e78cb9c0',
 'categorical__maritalstatus_703M_ea6fb4b5']

In [4]:
# No NaNs left, and scaled columns are ~mean 0 / std 1 on the fit period.
assert not X_fit.isna().any().any() and not X_test.isna().any().any()
X_fit.filter(like="numeric__credit_to_income").describe().round(2)

,numeric__credit_to_income
count,869559.00
mean,0.00
std,1.00
min,-1.28
25%,-0.65
50%,-0.26
75%,0.33
max,9.26


## 3. Save model-ready tables (with `case_id`, `target`, `WEEK_NUM`, `thin_file` kept alongside for evaluation)

In [5]:
def save(X, source, name):
    keep = [c for c in ["case_id", "target", "WEEK_NUM", "thin_file"] if c in source]
    out = pd.concat([source[keep].reset_index(drop=True), X.astype("float32").reset_index(drop=True)], axis=1)
    out.to_parquet(PROCESSED / f"model_ready_{name}.parquet", index=False)
    return out.shape


for X, source, name in [(X_fit, fit_df, "fit"), (X_valid, valid_df, "valid"), (X_test, test, "test")]:
    print(name, save(X, source, name))

fit (869559, 110)


valid (130441, 110)


test (200000, 109)


## 4. Does everything connect? A baseline logistic regression

Not the final model (that's October). This checks that the pipeline produces usable inputs, and it compares
**Experiment A** (application fields only) against **Experiment B** (+ engineered history features) from the technical report (§16).

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

STATIC_ONLY = ["mainoccupationinc_384A", "credamount_770A", "annuity_780A", "days_employed_700P",
               "education_927M", "maritalstatus_703M", "case_id", "target"]


def evaluate(X_tr, X_va, label):
    model = LogisticRegression(max_iter=1000).fit(X_tr, fit_df["target"])
    p = model.predict_proba(X_va)[:, 1]
    y, thin = valid_df["target"].to_numpy(), valid_df["thin_file"].to_numpy() == 1
    return {"experiment": label, "features": X_tr.shape[1],
            "AUC": roc_auc_score(y, p), "LogLoss": log_loss(y, p), "Brier": brier_score_loss(y, p),
            "AUC thin-file": roc_auc_score(y[thin], p[thin]), "AUC established": roc_auc_score(y[~thin], p[~thin])}


Xa_fit, pre_a = encode_scale(fit_df[STATIC_ONLY])
Xa_valid, _ = encode_scale(valid_df[STATIC_ONLY], preprocessor=pre_a)

pd.DataFrame([
    evaluate(Xa_fit, Xa_valid, "A: static_0 only"),
    evaluate(X_fit, X_valid, "B: + engineered features"),
]).set_index("experiment").round(4)

,features,AUC,LogLoss,Brier,AUC thin-file,AUC established
experiment,,,,,,
A: static_0 only,17,0.7378,0.4095,0.1280,0.7066,0.7540
B: + engineered features,106,0.7597,0.3980,0.1241,0.7082,0.7856


**Reading the result:** AUC measures ranking (higher is better), while LogLoss and Brier measure probability quality (lower is better).
If B beats A, the history features you built add real signal on *later* weeks the model never saw.

Next (October): random forest / gradient boosting on `model_ready_*.parquet`, calibration, then the profit and inclusion-gap policy layer.